# Simplified No-Optimization CG-Projection Enclosure for the 30k Beam System

This notebook applies the CG-projection enclosure from `system_303_solver_v3_simplified.ipynb` to `wing_bsm_30k.pkl`. It uses symmetric Jacobi scaling, removes the first three boundary-condition coordinates, runs exactly 520 CG iterations, and evaluates the projection contraction every 10 iterations.

There is no relaxation term: the projected radius is exactly

$$
d_m^{\mathrm{proj}} = |\Pi_m|d^0.
$$

The reduced beam system has 30,000 coordinates. The dense matrix stored in the pickle is converted immediately to sparse CSR form, and only evenly sampled projector rows are retained for diagnostics; this avoids materializing a 30,000-by-30,000 dense projector while preserving the same row-wise projection calculation. The two supplied boxes are identical after the three boundary-condition coordinates are removed, so `B0_1` is used.

In [1]:
from pathlib import Path
import gc
import pickle

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from scipy.sparse import csr_matrix, diags


pickle_path = Path("wing_bsm_30k.pkl")

with pickle_path.open("rb") as file:
    K_full, F_full, U_full, B0_1_full, B0_2_full = pickle.load(file)

# Remove the first three boundary-condition coordinates, as in v3.
A = csr_matrix(K_full[3:, 3:])
b = np.asarray(F_full[3:], dtype=float).copy()
x_star = np.asarray(U_full[3:], dtype=float).copy()
box = np.asarray(B0_1_full[3:], dtype=float).copy()
boxes_match_after_bc = np.array_equal(B0_1_full[3:], B0_2_full[3:])

# Release the 6.7 GiB dense serialization after creating the sparse matrix.
del K_full, F_full, U_full, B0_1_full, B0_2_full
gc.collect()

n = len(b)
if A.shape != (n, n):
    raise ValueError(f"Expected A to have shape {(n, n)}, got {A.shape}.")
if box.shape != (n, 2):
    raise ValueError(f"Expected box to have shape {(n, 2)}, got {box.shape}.")
if np.any(box[:, 0] > box[:, 1]):
    raise ValueError("The initial box has lower bounds above upper bounds.")

print(f"system dimension: {n}")
print(f"sparse matrix nonzeros: {A.nnz}")
print(f"B0_1 and B0_2 match after removing BC coordinates: {boxes_match_after_bc}")
print(f"stored solution relative residual: {np.linalg.norm(b - A @ x_star) / np.linalg.norm(b):.6e}")

system dimension: 30000
sparse matrix nonzeros: 129994
B0_1 and B0_2 match after removing BC coordinates: True
stored solution relative residual: 1.254917e-01


## Jacobi scaling

With `scale = sqrt(diag(A))`, solve in the coordinates $y=\mathrm{scale}\,x$ and transform the reported enclosure coordinates back to the original variables.

In [2]:
diagonal = A.diagonal()
if np.any(diagonal <= 0):
    raise ValueError("Jacobi scaling requires a positive diagonal.")

scale = np.sqrt(diagonal)
inverse_scale = 1.0 / scale
A_hat = diags(inverse_scale) @ A @ diags(inverse_scale)
b_hat = inverse_scale * b
ell0 = box[:, 0]
u0 = box[:, 1]
ell_hat = scale * ell0
u_hat = scale * u0
c0 = 0.5 * (ell_hat + u_hat)

max_iterations = 520
contraction_period = 10
max_plot_coordinates = 80

all_coordinates = np.arange(n)
all_plot_indices = np.unique(
    np.linspace(0, n - 1, max_plot_coordinates).round().astype(int)
)
mod1_coordinates = all_coordinates[all_coordinates % 3 == 1]
mod1_plot_indices = np.unique(
    mod1_coordinates[
        np.linspace(0, len(mod1_coordinates) - 1, max_plot_coordinates)
        .round()
        .astype(int)
    ]
)
tracked_indices = np.unique(np.concatenate([all_plot_indices, mod1_plot_indices]))

print(f"tracked projector rows: {len(tracked_indices)}")
print(f"CG iterations: {max_iterations}")
print(f"contraction period: {contraction_period}")

tracked projector rows: 134
CG iterations: 520
contraction period: 10


## CG and projection contraction

The function below uses the same fixed-iteration CG recurrence as the source notebook. Because this much larger and more ill-conditioned beam loses finite-precision CG conjugacy, every recorded direction is explicitly reorthogonalized in the $A$ inner product. The resulting basis $Q_m$ spans the recorded directions and satisfies $Q_m^T A Q_m\approx I$, giving

$$
\widehat x_m=c^0+Q_mQ_m^T(b-Ac^0),\qquad
\Pi_m=I-Q_mQ_m^TA.
$$

Only the requested rows of $\Pi_m$ are stored. At every tenth iteration, those rows directly evaluate the interval hull $|\Pi_m|d^0$ for the tracked coordinates.

In [3]:
def conjugate_gradient_projection_history(
    A,
    b,
    ell0,
    u0,
    coordinate_indices,
    iterations,
    period,
    orthogonality_tolerance=1e-12,
):
    """Run fixed-iteration CG and contract rows using a stable A-orthonormal basis."""
    ell0 = np.asarray(ell0, dtype=float)
    u0 = np.asarray(u0, dtype=float)
    coordinate_indices = np.asarray(coordinate_indices, dtype=int)

    c0 = 0.5 * (ell0 + u0)
    d0 = 0.5 * (u0 - ell0)
    initial_residual = b - A @ c0

    x_cg = c0.copy()
    r = initial_residual.copy()
    p = r.copy()
    x_projection = c0.copy()

    # Preallocate the stable direction basis and its A-products.
    Q = np.empty((len(b), iterations), dtype=float)
    AQ = np.empty((len(b), iterations), dtype=float)
    rank = 0

    projector_rows = np.zeros((len(coordinate_indices), len(b)), dtype=float)
    projector_rows[np.arange(len(coordinate_indices)), coordinate_indices] = 1.0

    iteration_history = [0]
    residual_norm_history = [float(np.linalg.norm(r))]
    rank_history = [0]
    ell_history = [ell0[coordinate_indices].copy()]
    u_history = [u0[coordinate_indices].copy()]

    for k in range(iterations):
        Ap = A @ p
        rr = float(r @ r)
        denominator = float(p @ Ap)
        if denominator <= 0 or not np.isfinite(denominator):
            raise RuntimeError(f"Invalid CG denominator at iteration {k}.")

        # Twice-reorthogonalized modified Gram-Schmidt in the A inner product.
        q_candidate = p.copy()
        Aq_candidate = Ap.copy()
        for _ in range(2):
            if rank:
                coefficients = Q[:, :rank].T @ Aq_candidate
                q_candidate -= Q[:, :rank] @ coefficients
                Aq_candidate -= AQ[:, :rank] @ coefficients

        candidate_norm_squared = float(q_candidate @ Aq_candidate)
        if (
            np.isfinite(candidate_norm_squared)
            and candidate_norm_squared > orthogonality_tolerance * denominator
        ):
            candidate_norm = np.sqrt(candidate_norm_squared)
            q = q_candidate / candidate_norm
            Aq = Aq_candidate / candidate_norm
            Q[:, rank] = q
            AQ[:, rank] = Aq
            rank += 1

            x_projection += q * float(q @ initial_residual)
            projector_rows -= q[coordinate_indices, None] * Aq[None, :]

        alpha = rr / denominator
        x_cg = x_cg + alpha * p
        r_next = r - alpha * Ap
        iteration = k + 1

        if iteration % period == 0:
            d_projected = np.abs(projector_rows) @ d0
            x_hat = x_projection[coordinate_indices]
            ell = np.maximum(ell0[coordinate_indices], x_hat - d_projected)
            u = np.minimum(u0[coordinate_indices], x_hat + d_projected)
            if np.any(ell > u):
                inverted_coordinates = coordinate_indices[ell > u]
                raise RuntimeError(
                    "The projection produced an inverted interval at iteration "
                    f"{iteration}; first affected coordinate: {inverted_coordinates[0]}."
                )

            iteration_history.append(iteration)
            residual_norm_history.append(float(np.linalg.norm(r_next)))
            rank_history.append(rank)
            ell_history.append(ell)
            u_history.append(u)

        if k < iterations - 1:
            beta = float(r_next @ r_next) / rr
            p = r_next + beta * p
        r = r_next

    return {
        "x_cg": x_cg,
        "x_projection": x_projection,
        "iterations": np.asarray(iteration_history, dtype=int),
        "residual_norms": np.asarray(residual_norm_history),
        "rank_history": np.asarray(rank_history, dtype=int),
        "ell_history": np.asarray(ell_history),
        "u_history": np.asarray(u_history),
    }

## Run the enclosure


In [4]:
result = conjugate_gradient_projection_history(
    A=A_hat,
    b=b_hat,
    ell0=ell_hat,
    u0=u_hat,
    coordinate_indices=tracked_indices,
    iterations=max_iterations,
    period=contraction_period,
)

iteration_history = result["iterations"]
ell_history = result["ell_history"] / scale[tracked_indices][None, :]
u_history = result["u_history"] / scale[tracked_indices][None, :]
ell_final = ell_history[-1]
u_final = u_history[-1]

initial_width = u0[tracked_indices] - ell0[tracked_indices]
width_history = u_history - ell_history
with np.errstate(divide="ignore", invalid="ignore"):
    shrinkage_history = 100.0 * (
        1.0 - width_history / initial_width[None, :]
    )

x_cg = result["x_cg"] / scale
x_projection = result["x_projection"] / scale
cg_relative_residual = np.linalg.norm(b - A @ x_cg) / np.linalg.norm(b)
projection_relative_residual = np.linalg.norm(b - A @ x_projection) / np.linalg.norm(b)
contains_stored_solution = np.all(
    (ell_final <= x_star[tracked_indices])
    & (x_star[tracked_indices] <= u_final)
)

print(f"CG iterations: {max_iterations}")
print(f"stable direction rank: {result['rank_history'][-1]}")
print(f"contraction calls: {len(iteration_history) - 1}")
print(f"final CG relative residual: {cg_relative_residual:.6e}")
print(f"final projection-center relative residual: {projection_relative_residual:.6e}")
print(f"final tracked mean shrinkage: {np.nanmean(shrinkage_history[-1]):.2f}%")
print(f"tracked final enclosure contains stored U: {contains_stored_solution}")

CG iterations: 520
stable direction rank: 520
contraction calls: 52
final CG relative residual: 6.206375e+09
final projection-center relative residual: 1.593178e+15
final tracked mean shrinkage: 0.00%
tracked final enclosure contains stored U: False


## Controllable animations

Each animation includes play/pause buttons and a frame slider. To keep the 30,000-coordinate projection tractable, the plots use evenly sampled coordinate rows. Coordinate indices use Python's zero-based convention, so the `1 mod 3` coordinates are `1, 4, 7, ...`.

In [5]:
def tracked_positions(indices):
    positions = np.searchsorted(tracked_indices, indices)
    if not np.array_equal(tracked_indices[positions], indices):
        raise ValueError("Every plotted coordinate must be a tracked projector row.")
    return positions


def display_shrinkage_animation(indices, title):
    positions = tracked_positions(indices)
    frame_data = shrinkage_history[:, positions]
    finite_data = frame_data[np.isfinite(frame_data)]
    y_min = min(0.0, float(np.min(finite_data))) if finite_data.size else 0.0
    y_max = max(1.0, float(np.max(finite_data)) * 1.05) if finite_data.size else 1.0
    padding = 0.03 * max(y_max - y_min, 1.0)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    line, = ax.plot(indices, np.nan_to_num(frame_data[0]), linewidth=1)
    label = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(xlim=(indices[0], indices[-1]), ylim=(y_min - padding, y_max + padding))
    ax.set_xlabel("Coordinate index")
    ax.set_ylabel("Width reduction (% of original width)")
    ax.set_title(title)
    ax.grid(alpha=0.25)

    def update(frame):
        line.set_ydata(np.nan_to_num(frame_data[frame]))
        label.set_text(f"CG iteration {iteration_history[frame]}")
        return line, label

    animation = FuncAnimation(fig, update, frames=len(iteration_history), interval=180)
    display(HTML(animation.to_jshtml(fps=5, default_mode="loop")))
    plt.close(fig)
    return animation


def display_bounds_animation(indices, title):
    positions = tracked_positions(indices)
    lower_data = ell_history[:, positions]
    upper_data = u_history[:, positions]
    y_min = float(np.min(lower_data))
    y_max = float(np.max(upper_data))
    padding = 0.03 * max(y_max - y_min, 1.0)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    lower_line, = ax.plot(indices, lower_data[0], label="lower bound", linewidth=1)
    upper_line, = ax.plot(indices, upper_data[0], label="upper bound", linewidth=1)
    label = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")
    ax.set(xlim=(indices[0], indices[-1]), ylim=(y_min - padding, y_max + padding))
    ax.set_xlabel("Coordinate index")
    ax.set_ylabel("Bound value")
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend()

    def update(frame):
        lower_line.set_ydata(lower_data[frame])
        upper_line.set_ydata(upper_data[frame])
        label.set_text(f"CG iteration {iteration_history[frame]}")
        return lower_line, upper_line, label

    animation = FuncAnimation(fig, update, frames=len(iteration_history), interval=180)
    display(HTML(animation.to_jshtml(fps=5, default_mode="loop")))
    plt.close(fig)
    return animation

### Shrinkage along sampled coordinates

In [6]:
all_coordinate_animation = display_shrinkage_animation(
    all_plot_indices,
    "Shrinkage along sampled coordinates",
)

### Shrinkage along sampled coordinates 1 mod 3

In [7]:
mod1_shrinkage_animation = display_shrinkage_animation(
    mod1_plot_indices,
    "Shrinkage along sampled coordinates 1 mod 3",
)

### Upper and lower bounds for sampled coordinates 1 mod 3

In [8]:
mod1_bounds_animation = display_bounds_animation(
    mod1_plot_indices,
    "Upper and lower bounds for sampled coordinates 1 mod 3",
)